### Cell 1: Import các thư viện cần thiết


In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import gc

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)


### Cell 2: Đọc dữ liệu từ toàn bộ các năm (2016-2024)


In [ ]:
# Đường dẫn tới thư mục chứa dữ liệu của tất cả các năm
data_path = '../data/processed/tabular_by_year'

print("Đang tải dữ liệu toàn bộ các năm (2016-2024)...")
# Pandas có thể tự động đọc toàn bộ các thư mục con phân vùng (partition)
df = pd.read_parquet(data_path)

print(f"Kích thước tập dữ liệu gốc: {df.shape}")
print("Phân phối dữ liệu theo năm:")
print(df['source_year'].value_counts().sort_index())


Đang tải dữ liệu toàn bộ các năm (2016-2024)...


### Cell 3: Feature Engineering chung & Xử lý Missing Values
Thực hiện xử lý thời gian và điền dữ liệu khuyết cho toàn bộ tập dữ liệu trước.


In [ ]:
# 1. Chuyển đổi thời gian thành giờ
df['CRS_DEP_HOUR'] = pd.to_datetime(df['CRS_DEP_TIME'], errors='coerce').dt.hour
df['CRS_ARR_HOUR'] = pd.to_datetime(df['CRS_ARR_TIME'], errors='coerce').dt.hour

# 2. Xử lý missing values cho thời tiết bằng trung vị (median)
weather_cols = ['O_TEMP', 'O_PRCP', 'O_WSPD', 'D_TEMP', 'D_PRCP', 'D_WSPD']
imputer = SimpleImputer(strategy='median')
df[weather_cols] = imputer.fit_transform(df[weather_cols])

# 3. Label Encoding cho biến Categorical
categorical_cols = ['OP_CARRIER', 'ORIGIN', 'DEST']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Loại bỏ NaN ở các cột còn lại
df = df.dropna(subset=['CRS_ELAPSED_TIME', 'CRS_DEP_HOUR', 'CRS_ARR_HOUR', 'ARR_DELAY', 'DEP_DELAY'])

print(f"Kích thước dữ liệu sau khi tiền xử lý chung: {df.shape}")


### Cell 4: Hàm chuẩn bị dữ liệu và huấn luyện mô hình
Hàm này cho phép tạo 2 mô hình khác nhau (dự đoán DEP_DELAY và ARR_DELAY) với cấu hình biến (features) tùy chỉnh.


In [ ]:
def prepare_and_train(df_full, target_col, exclude_cols, drop_weather_dest=False, model_types=['xgb', 'lr', 'rf']):
    print(f"{'='*50}")
    print(f"BẮT ĐẦU HUẤN LUYỆN CÁC MÔ HÌNH DỰ ĐOÁN: {target_col} >= 15 phút")
    print(f"CÁC MÔ HÌNH ĐƯỢC CHỌN: {model_types}")
    print(f"{'='*50}")
    
    df_model = df_full.copy()
    
    target_name = f'TARGET_{target_col}'
    df_model[target_name] = (df_model[target_col] >= 15).astype(int)
    
    cols_to_drop = exclude_cols + ['FL_DATE', 'flight_key', 'OP_CARRIER_FL_NUM', 'source_row_number', 'FLIGHTS', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 'ARR_DELAY', 'DEP_DELAY']
    
    if drop_weather_dest:
        print("-> Đang loại bỏ các trường thời tiết ở nơi đến (D_TEMP, D_PRCP, D_WSPD)...")
        cols_to_drop += ['D_TEMP', 'D_PRCP', 'D_WSPD', 'D_LATITUDE', 'D_LONGITUDE', 'DEST_INDEX']
        
    df_model = df_model.drop(columns=cols_to_drop, errors='ignore')
    
    train_mask = df_model['source_year'].between(2016, 2022)
    valid_mask = df_model['source_year'] == 2023
    test_mask = df_model['source_year'] == 2024
    
    X_train = df_model[train_mask].drop(columns=[target_name, 'source_year'])
    y_train = df_model[train_mask][target_name]
    
    X_valid = df_model[valid_mask].drop(columns=[target_name, 'source_year'])
    y_valid = df_model[valid_mask][target_name]
    
    X_test = df_model[test_mask].drop(columns=[target_name, 'source_year'])
    y_test = df_model[test_mask][target_name]
    
    print(f"Kích thước Train (2016-2022): {X_train.shape}")
    print(f"Kích thước Valid (2023)    : {X_valid.shape}")
    print(f"Kích thước Test (2024)     : {X_test.shape}")
    
    del df_model
    gc.collect()
    
    numeric_cols = [col for col in X_train.columns if col not in ['OP_CARRIER', 'ORIGIN', 'DEST', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK']]
    scaler = StandardScaler()
    
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_valid[numeric_cols] = scaler.transform(X_valid[numeric_cols])
    X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])
    
    trained_models = {}
    
    for m_type in model_types:
        print(f"{'-'*40}")
        print(f"Đang huấn luyện mô hình {m_type.upper()}...")
        if m_type == 'rf':
            clf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
        elif m_type == 'xgb':
            import xgboost as xgb
            clf = xgb.XGBClassifier(n_estimators=100, max_depth=10, random_state=42, tree_method='hist', device='cuda')
        elif m_type == 'lr':
            try:
                from sklearn.linear_model import LogisticRegression
                print("Sử dụng cuML Logistic Regression (GPU)...")
                clf = LogisticRegression(max_iter=1000)
            except ImportError:
                from sklearn.linear_model import LogisticRegression
                print("Không tìm thấy cuML, fallback về sklearn Logistic Regression (CPU)...")
                clf = LogisticRegression(max_iter=1000, n_jobs=-1)
        else:
            print(f"Bỏ qua {m_type}: model_type không hợp lệ.")
            continue
        
        clf.fit(X_train, y_train)
        
        print(f"Đang đánh giá {m_type.upper()} trên tập Validation (2023)...")
        y_val_pred = clf.predict(X_valid)
        y_val_prob = clf.predict_proba(X_valid)[:, 1]
        print(f"Validation ROC-AUC: {roc_auc_score(y_valid, y_val_prob):.4f}")
        
        print(f"Đang dự đoán và đánh giá {m_type.upper()} trên tập Test (2024)...")
        y_test_pred = clf.predict(X_test)
        y_test_prob = clf.predict_proba(X_test)[:, 1]
        
        print(f"--- Báo cáo kết quả phân loại {m_type.upper()} (Test 2024) ---")
        print(classification_report(y_test, y_test_pred))
        print(f"Test ROC-AUC Score: {roc_auc_score(y_test, y_test_prob):.4f}")
        
        trained_models[m_type] = clf
        
    return trained_models


### Cell 5: Mô hình 1 - Dự đoán Khởi hành trễ (DEP_DELAY >= 15)
Đối với dự đoán khởi hành trễ, chúng ta không biết các thông tin diễn ra trong và sau khi bay (như Taxi Out/In, Wheels Off/On).


In [ ]:
# Các cột không được sử dụng khi dự đoán Khởi hành trễ
leakage_dep = [
    'DEP_TIME', 'TAXI_OUT', 'WHEELS_OFF', 
    'WHEELS_ON', 'TAXI_IN', 'ARR_TIME', 
    'ACTUAL_ELAPSED_TIME', 'AIR_TIME'
]

models_dep = prepare_and_train(
    df, 
    target_col='DEP_DELAY', 
    exclude_cols=leakage_dep, 
    drop_weather_dest=False,
    model_types=['xgb', 'lr', 'rf']
)


### Cell 6: Mô hình 2 - Dự đoán Đến trễ (ARR_DELAY >= 15)
Đối với dự đoán đến trễ, **nếu** dự đoán được thực hiện trước khi cất cánh, chúng ta không thể dùng DEP_DELAY. **Tuy nhiên**, nếu bài toán là dự đoán sau khi đã cất cánh, ta có thể đưa DEP_DELAY vào.\nTheo yêu cầu của bạn: Không sử dụng các trường thời tiết ở nơi đến (`D_*`) để dự đoán.


In [ ]:
# Các cột rò rỉ dữ liệu cho Arrival Delay (nếu giả sử dự đoán từ lúc trước khi bay)
# Nếu bạn muốn dùng DEP_DELAY làm biến dự đoán cho ARR_DELAY (tức là dự đoán trên không), hãy bỏ 'DEP_DELAY' khỏi danh sách này.
leakage_arr = [
    'DEP_TIME', 'TAXI_OUT', 'WHEELS_OFF', 
    'WHEELS_ON', 'TAXI_IN', 'ARR_TIME', 
    'ACTUAL_ELAPSED_TIME', 'AIR_TIME'
]

models_arr = prepare_and_train(
    df, 
    target_col='ARR_DELAY', 
    exclude_cols=leakage_arr, 
    drop_weather_dest=True,  # Loại bỏ D_TEMP, D_PRCP, D_WSPD theo yêu cầu
    model_types=['xgb', 'lr', 'rf']
)
